# Curvature-Aware GNN Exploration

This notebook walks through the main steps: dataset exploration, Ollivier-Ricci curvature on a subgraph, bottleneck visualization, and before/after rewiring comparison.

## 1. Setup and dataset loading

We load Cora from PyTorch Geometric and convert to NetworkX for curvature computation. Cora is a citation network: nodes are papers, edges are citations, and we predict the paper's topic (class).

In [ ]:
%pip install -q -r requirements.txt
import sys
import os
# Avoid OpenMP/NetworKit crash on macOS (multiple libomp runtimes). Set before other imports.
os.environ.setdefault("OMP_NUM_THREADS", "1")
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
os.chdir(ROOT)

from src.utils import load_dataset, pyg_to_networkx, get_graph_stats, CONFIG
import logging
logging.basicConfig(level=logging.INFO)

data = load_dataset("Cora", root=os.path.join(ROOT, "data"))
G = pyg_to_networkx(data)
get_graph_stats(G)
print("Nodes:", data.num_nodes, "Edges:", data.edge_index.size(1), "Features:", data.num_node_features, "Classes:", data.y.max().item() + 1)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
Note: you may need to restart the kernel to use updated packages.


INFO:src.utils:Loaded Cora: 2708 nodes, 10556 edges, 1433 features, 7 classes
INFO:src.utils:Graph stats: nodes=2708, edges=5278, avg_degree=3.90, diameter(LCC)=19, avg_clustering=0.2407, num_components=78


Nodes: 2708 Edges: 10556 Features: 1433 Classes: 7


: 

## 2. Theory: Over-squashing and curvature

**Over-squashing** (Topping et al., 2022) occurs when information from many nodes is compressed through few edges (bottlenecks), limiting GNN expressiveness. **Ollivier-Ricci curvature** assigns a scalar to each edge: negative curvature indicates a bottleneck (geodesics converge), while positive curvature indicates dense local structure. We compute it on a small subgraph next to keep runtime low.

In [ ]:
import networkx as nx
import numpy as np

# Take a small connected subgraph (e.g. around node 0)
nodes_0 = list(nx.ego_graph(G, 0, radius=2).nodes())
G_small = G.subgraph(nodes_0).copy()
print("Subgraph nodes:", G_small.number_of_nodes(), "edges:", G_small.number_of_edges())

from src.curvature import compute_ollivier_ricci, get_node_curvature

G_curved, edge_curvatures = compute_ollivier_ricci(G_small, alpha=0.5)
node_curv = get_node_curvature(G_curved)
print("Sample edge curvatures:", list(edge_curvatures.items())[:5])
print("Mean node curvature (sample):", np.mean(list(node_curv.values())))

Subgraph nodes: 8 edges: 10


## 3. Visualizing bottleneck edges

Edges with negative curvature (red/dark in a diverging colormap) are bottlenecks. We plot the subgraph with nodes colored by mean curvature (RdYlGn: green = higher curvature, red = lower).

In [ ]:
from src.visualization import plot_graph_curvature

plot_graph_curvature(G_curved, node_curv, "Cora_subgraph", sample_size=500)
print("Saved to figures/graph_curvature_Cora_subgraph.png")

## 4. Curvature-based rewiring

**Rewiring** adds shortcut edges between bottleneck-adjacent nodes that are 2 hops apart, reducing the squeeze through negative-curvature edges. Here we compare edge count and then train a small GCN on original vs rewired graph (quick run).

In [ ]:
from src.rewiring import curvature_based_rewiring, random_rewiring
from src.models import GCN, train_model, evaluate_model
import torch

# Full graph curvature (on LCC) for rewiring
from src.curvature import compute_ollivier_ricci
G_full_curved, edge_curv_full = compute_ollivier_ricci(G, alpha=CONFIG["curvature_alpha"])

data_curv = curvature_based_rewiring(G_full_curved, edge_curv_full, data, threshold=CONFIG["rewiring_threshold"], num_edges_add=CONFIG["num_edges_add"])
data_rand = random_rewiring(data, num_edges_add=CONFIG["num_edges_add"], seed=42)

print("Original edges:", data.edge_index.size(1)//2, "Curvature-rewired:", data_curv.edge_index.size(1)//2, "Random-rewired:", data_rand.edge_index.size(1)//2)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_g = data.to(device)
data_curv_g = data_curv.to(device)

num_features = data.num_node_features
num_classes = int(data.y.max().item()) + 1

for name, d in [("Original", data_g), ("Curvature-rewired", data_curv_g)]:
    model = GCN(num_features, num_classes, hidden_dim=CONFIG["hidden_dim"], num_layers=CONFIG["num_gnn_layers"], dropout=CONFIG["dropout"]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
    train_model(model, d, opt, epochs=50, device=device)
    train_acc, val_acc, test_acc, _ = evaluate_model(model, d, device)
    print(f"{name}: Test accuracy = {test_acc:.4f}")

## 5. Summary

Curvature-based rewiring adds shortcuts at bottleneck regions (negative Ollivier-Ricci curvature), which typically improves test accuracy compared to the original graph and to random rewiring. Run `main.py` for full experiments with multiple runs and all plots.